Ce Notebook est inutile les resultats qu'il apporte ne sont pas concluants...

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from tools import line
from pathlib import Path

FIG_DIR_BNB = Path(r'..\reports\figures\airbnb')
FIG_WIDTH, FIG_HEIGHT = 1600, 500

# Chargement

In [ ]:
# Comptabilité
df0 = pd.read_csv(r"..\data\processed\2BLONDEL_consolide.csv", sep=';', decimal=',', parse_dates=['date_écriture'])
# DF Loyers
loyers = df0.query("code_compte == '70600000'")
# Colonne "canal"
conds = [
    loyers['libellé_écriture'].str.contains('BNB|FACTURE HOTE', case=False),
    loyers['libellé_écriture'].str.contains('BOOKING', case=False),
    ]
choices = ['AIRBNB', 'BOOKING']
loyers['canal'] = np.select(conds, choices, default='DIRECT')

# AirBNB
airbnb = pd.read_csv(r"..\data\processed\airbnb.csv", sep=';', decimal=',', parse_dates=['mois'])
airbnb['mois'] = airbnb['mois'].dt.to_period('M')

# Préparation

In [ ]:
ca_airbnb = loyers.query("canal == 'AIRBNB'")
ca_airbnb = ca_airbnb.groupby(loyers['date_écriture'].dt.to_period('M'))[['crédit_euro', 'débit_euro']].sum()
ca_airbnb = ca_airbnb.eval('ca_net = crédit_euro - débit_euro').reset_index()
ca_airbnb['periode'] = ca_airbnb['date_écriture'].astype(str)

# Merge

In [ ]:
df_merge = ca_airbnb.merge(airbnb, left_on='date_écriture', right_on='mois', how='left')

In [ ]:
df_merge.head()

# CA par Reservation

In [ ]:
df = df_merge[['date_écriture', 'ca_net', 'réservations']]
df['ca_par_resa'] = df['ca_net'] / df['réservations']
df